# Member 1 - Logistic Regression Model

## Objective
The goal of this notebook is to train and evaluate a Logistic Regression model for the Obesity dataset.


In [27]:
import os
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from shared.preprocessing.preprocess import load_data, split_features_target, get_feature_types
from shared.evaluation.metrics import evaluate_model, print_evaluation_results

In [28]:
import os

print("Current working directory:", os.getcwd())

file_path = os.path.join(project_root, "data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")
print("File exists:", os.path.exists(file_path))

Current working directory: d:\Machine Learning\Machine-Learning---Final-Project\member1\notebook
File exists: False


# Load the raw dataset

The raw CSV is loaded first, and then the features and target are separared.

In [29]:
file_path = os.path.join(project_root, "data", "raw", "ObesityDataSet_raw_and_data_sinthetic.csv")
df = load_data(file_path)
X, y = split_features_target(df, target_column="NObeyesdad")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/ObesityDataSet_raw_and_data_sinthetic.csv'

In [ ]:
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
numeric_features, categorical_features = get_feature_types(X)
print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

Dataset columns:
['Gender', 'Age', 'Height', 'Weight', 'family_history_with_overweight', 'FAVC', 'FCVC', 'NCP', 'CAEC', 'SMOKE', 'CH2O', 'SCC', 'FAF', 'TUE', 'CALC', 'MTRANS', 'NObeyesdad']


# Build the model-specific preprocessing pipeline

For Logistic Regression, numeric features are scaled and categorical features are one-hot encoded.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=3000))
])

param_grid = {
    'model__C': [0.1, 1.0, 10.0],
    'model__solver': ['lbfgs']
}

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

Logistic Regression model trained successfully.


c:\Users\PV\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Train the pipeline with GridSearchCV
This follows the professor's instruction that GridSearch should be attached directly to the model pipeline.

In [ ]:
grid_search.fit(X_train, y_train)
print('Best Parameters:', grid_search.best_params_)
print('Best Cross-Validation Score:', round(grid_search.best_score_, 4))

Accuracy: 0.8085106382978723

Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.83      0.84        54
           1       0.63      0.71      0.67        58
           2       0.82      0.84      0.83        70
           3       0.88      1.00      0.94        60
           4       1.00      0.98      0.99        65
           5       0.75      0.72      0.74        58
           6       0.69      0.53      0.60        58

    accuracy                           0.81       423
   macro avg       0.80      0.80      0.80       423
weighted avg       0.81      0.81      0.81       423


Confusion Matrix:
[[45  9  0  0  0  0  0]
 [ 8 41  0  0  0  7  2]
 [ 0  0 59  5  0  1  5]
 [ 0  0  0 60  0  0  0]
 [ 0  0  0  1 64  0  0]
 [ 0  7  2  0  0 42  7]
 [ 0  8 11  2  0  6 31]]


# Evaluate the best model

In [ ]:
best_model = grid_search.best_estimator_
results = evaluate_model(best_model, X_test, y_test)
print_evaluation_results(results)
print(f'\nTest Accuracy: {results["accuracy"]:.4f}')

Accuracy: 0.8085


## Result Interception
The Logistic Regression model is used as a baseline classification model.

### Strengths
- simple and fast
- easy to train
- easy to explain
- good as a baseline model for comparison

### Weaknesses
- may not capture complex nonlinear relationships
- may perform worse than ensemble models such as Random Forest